# 60 — Check a USDM study definition against the published model

Point `STUDY_JSON` below at any USDM v4 study definition (the JSON that
DDF-compliant systems exchange) and run all cells. Three checks:

1. **Context conformance** — every `instanceType` is a model class, every
   serialization key resolves to a model attribute, every id-based
   cross-reference resolves to an object in the document.
2. **Structural SHACL** (`usdm_v4.shapes.ttl`) — cardinality, datatypes,
   ranges, closedness.
3. **Terminology SHACL** (`usdm_v4.shapes-ct.ttl`) — the 25 DDF-native
   value sets, severity from CDISC's published extensibility flags.

What this is **not**: a regulatory or protocol-quality judgment.
Terminology coverage is the DDF-native value sets only — codelists backed
by external terminology packages (SDTM/Protocol Terminology, MedDRA,
ISO, ...) pass no value check here. See `../docs/shacl-design.md`.

This notebook reports findings; it asserts nothing about them. It is an
optional checker, not part of the generation pipeline — the pipeline's
regression baselines live in `30_validate.ipynb`.

Requires the deliverables at the repo root (run notebooks 20, 40, 50
first, or use a release checkout) and the default input requires
`30_validate.ipynb` to have fetched the pilot study once.


## 1. Input


In [1]:
import json
from pathlib import Path

STUDY_JSON = "../downloads/CDISC_Pilot_Study.json"   # <-- point this at your own file

if not Path(STUDY_JSON).exists():
    raise FileNotFoundError(
        f"{STUDY_JSON} not found — for the default pilot study, run"
        " notebooks/30_validate.ipynb once, or set STUDY_JSON to your own file."
    )

with open(STUDY_JSON) as f:
    document = json.load(f)

if "study" not in document:
    raise ValueError("expected the USDM API wrapper with a 'study' key")
usdm_version = document.get("usdmVersion")
if usdm_version != "4.0.0":
    raise ValueError(
        f"usdmVersion is {usdm_version!r} — this checker covers USDM 4.0.0 only"
    )
study = document["study"]

print(f"input:       {STUDY_JSON}")
print(f"usdmVersion: {usdm_version}")
print(f"system:      {document.get('systemName')} {document.get('systemVersion')}")


input:       ../downloads/CDISC_Pilot_Study.json
usdmVersion: 4.0.0
system:      CDISC USDM E2J 0.62.0


## 2. Context conformance


In [2]:
import rdflib
from rdflib.namespace import OWL, RDF

CONTEXT_PATH = "../usdm_v4.context.jsonld"
TTL_PATH = "../usdm_v4.ttl"
for p in (CONTEXT_PATH, TTL_PATH):
    if not Path(p).exists():
        raise FileNotFoundError(f"{p} missing — generate the deliverables first.")

with open(CONTEXT_PATH) as f:
    context = json.load(f)["@context"]

unknown_types = set()
unmapped = {}
null_ids = []
typed_objects = 0


def walk(node):
    global typed_objects
    if isinstance(node, dict):
        instance_type = node.get("instanceType")
        if isinstance(instance_type, str):
            typed_objects += 1
            if instance_type not in context:
                unknown_types.add(instance_type)
            else:
                scoped = context[instance_type]["@context"]
                for k in node:
                    if k not in scoped and k not in ("id", "instanceType"):
                        unmapped.setdefault(instance_type, set()).add(k)
            if node.get("id") is None:
                null_ids.append(instance_type)
        for k, v in node.items():
            if k != "@context":
                walk(v)
    elif isinstance(node, list):
        for v in node:
            walk(v)


walk(study)

study["@context"] = context
instances = rdflib.Graph()
instances.parse(
    data=json.dumps(study), format="json-ld",
    publicID=Path(STUDY_JSON).resolve().as_uri(),
)

ontology = rdflib.Graph()
ontology.parse(TTL_PATH, format="turtle")
from rdflib.namespace import DCTERMS

source = next(ontology.objects(
    rdflib.URIRef("https://w3id.org/cdisc/usdm/v4/"), DCTERMS.source), None)
if source is None or "/DDF-RA/" not in str(source):
    raise ValueError(
        "ontology header lacks a DDF-RA dcterms:source — cannot state what this check is pinned against"
    )
DDF_RA_TAG = str(source).split("/DDF-RA/")[1].split("/")[0]
print(f"checking against:     DDF-RA {DDF_RA_TAG} (from ontology dcterms:source)")
declared_props = (
    set(ontology.subjects(RDF.type, OWL.DatatypeProperty))
    | set(ontology.subjects(RDF.type, OWL.ObjectProperty))
)

ref_props = set()
for class_def in context.values():
    if isinstance(class_def, dict) and "@context" in class_def:
        for term in class_def["@context"].values():
            if term.get("@type") == "@id":
                ref_props.add(rdflib.URIRef(
                    term["@id"].replace("usdm:", "https://w3id.org/cdisc/usdm/v4/")
                ))

ref_links = 0
dangling = []
for p in ref_props:
    for s, o in instances.subject_objects(p):
        ref_links += 1
        if (o, RDF.type, None) not in instances:
            dangling.append((str(p).rsplit("/", 1)[1], str(o).rsplit("/", 1)[-1]))

print(f"typed objects:        {typed_objects}")
print(f"instance triples:     {len(instances)}")
print(f"unknown instanceTypes: {sorted(unknown_types) or 'none'}")
print(f"unmapped keys:        {({k: sorted(v) for k, v in unmapped.items()}) or 'none'}")
print(f"objects with null id: {null_ids or 'none'}")
print(f"cross-references:     {ref_links}, dangling: {len(dangling)}")
for prop, target in dangling[:20]:
    print(f"  dangling: {prop} -> {target}")


checking against:     DDF-RA v4.0.0 (from ontology dcterms:source)
typed objects:        1953
instance triples:     11811
unknown instanceTypes: none
unmapped keys:        none
objects with null id: ['Study']
cross-references:     1257, dangling: 0


## 3. SHACL — structural and terminology layers


In [3]:
import re
import pandas as pd
from pyshacl import validate
from rdflib.namespace import SH

SHAPES = {
    "structural": "../usdm_v4.shapes.ttl",
    "terminology": "../usdm_v4.shapes-ct.ttl",
}
for p in SHAPES.values():
    if not Path(p).exists():
        raise FileNotFoundError(f"{p} missing — run notebooks/50_generate_shapes.ipynb first.")

rows = []
for layer, path in SHAPES.items():
    shapes_graph = rdflib.Graph()
    shapes_graph.parse(path, format="turtle")
    conforms, report, _ = validate(
        data_graph=instances, shacl_graph=shapes_graph,
        inference="none", abort_on_first=False,
    )
    n = 0
    for result in report.subjects(SH.resultSeverity, None):
        n += 1
        path_value = report.value(result, SH.resultPath)
        focus = report.value(result, SH.focusNode)
        value = report.value(result, SH.value)
        value_str = str(value) if value is not None else ""
        rows.append({
            "layer": layer,
            "severity": str(report.value(result, SH.resultSeverity)).rsplit("#", 1)[1],
            "constraint": str(report.value(result, SH.sourceConstraintComponent)).rsplit("#", 1)[1].removesuffix("ConstraintComponent"),
            "path": str(path_value).rsplit("/", 1)[1] if path_value else "",
            "focus": str(focus).rsplit("/", 1)[-1],
            "value": value_str,
            "value_ncit": (f"http://purl.obolibrary.org/obo/NCIT_{value_str}"
                           if re.fullmatch(r"C\d+", value_str) else ""),
        })
    print(f"{layer:<12} conforms={conforms}  findings={n}")

findings = pd.DataFrame(rows, columns=["layer", "severity", "constraint", "path", "focus", "value", "value_ncit"])
if len(findings):
    display(findings.sort_values(["layer", "severity", "path"]).reset_index(drop=True))
else:
    print("no findings — the study conforms to both layers")


structural   conforms=True  findings=0
terminology  conforms=False  findings=14


,layer,severity,constraint,path,focus,value,value_ncit
0,terminology,Violation,In,Code-code,Code_10,C99905x2,
1,terminology,Violation,In,Code-code,Code_9,C99905x1,
2,terminology,Violation,In,Code-code,Code_11,C99905x3,
3,terminology,Warning,In,Code-code,Code_88,C99904x3,
4,terminology,Warning,In,Code-code,Code_1,C70793,http://purl.obolibrary.org/obo/NCIT_C70793
5,terminology,Warning,In,Code-code,Code_5,C70793,http://purl.obolibrary.org/obo/NCIT_C70793
6,terminology,Warning,In,Code-code,Code_157,C99907x1,
7,terminology,Warning,In,Code-code,Code_93,C99912x1,
8,terminology,Warning,In,Code-code,Code_673,C99903x1,
9,terminology,Warning,In,Code-code,Code_13,C132352,http://purl.obolibrary.org/obo/NCIT_C132352


## 4. Reading the results

- **Violation** (structural): the study breaks a model rule — a missing
  required attribute, a wrong datatype, an undeclared key, a value of the
  wrong class. `MinCount` on a `1..*` attribute with an empty list is the
  classic case: the information may exist in prose while the required
  structured form is empty.
- **Violation** (terminology): a value outside a **non-extensible**
  codelist. Three readings, in rough order of likelihood: the value is
  wrong; a placeholder was never replaced; or the value comes from a
  newer or parallel CT release than the pinned DDF-RA tag — the
  toolchain running ahead of the published standard (e.g. C207646
  "Study Acronym", CDISC-defined, where DDF-RA v4.0.0 publishes the
  ICH-rooted C94108 for the same term). Check the finding's code in
  NCIt before treating it as a data error.
- **Warning** (terminology): a value outside an **extensible** codelist.
  Sponsor extensions are legitimate use; the warning flags them for
  review, nothing more.
- The `focus` column carries the object's `id` from your JSON, so
  findings can be traced back to the source document.

Worked examples (DDF-RA v4.0.0 published studies): the CDISC Pilot is
structurally clean but carries 14 terminology findings — placeholder
codes (`C12345`, `C99905x1`, ...) that were never replaced. The Eli Lilly
diabetes example is the mirror image: every code valid, one structural
violation — an amendment whose two changes are described in the
`summary` prose while the required `changes` list (`1..*`) sits empty.


## Provenance

Part of [kerfors/usdm-rdf](https://github.com/kerfors/usdm-rdf). Checks
against the deliverables generated by notebooks 20, 40, and 50 from the
DDF-RA tag pinned in `notebooks/10_fetch_yaml.ipynb`.
